# 🎛 AutoDub Studio · TTS Models Tester

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syedj7895-cell/ai-video-dubber-AutoDub-Studio-Automatic-Dubbing-Engine-ElevenLabs-Quality-Open-Source-/blob/main/TTS_Tester.ipynb)

A lightweight, standalone playground to **compare TTS engines** for the
dubbing pipeline — especially **Hindi**. One tab per model; each model
downloads **only when you click Load** in its tab.

- Engines: Edge-TTS, gTTS, MMS, Piper, Kokoro, IndicF5, F5-TTS, Chatterbox,
  XTTS v2, CosyVoice 2/3, VibeVoice, VibeVoice-Hindi-7B, Veena, VEXYL, Qwen3-TTS
- No API keys, no payment (network engines: Edge/gTTS only)
- Fail-soft: errors appear in each tab's console


In [ ]:
# ── 1 ▸ Get the code (shares the AutoDub repo's tts_tester.py) ─────────
import pathlib, subprocess, sys

REPO_URL = ("https://github.com/syedj7895-cell/"
            "ai-video-dubber-AutoDub-Studio-Automatic-Dubbing-Engine-"
            "ElevenLabs-Quality-Open-Source-.git")
if not pathlib.Path("tts_tester.py").exists():
    if not pathlib.Path("ai-video-dubber").exists():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL,
                               "ai-video-dubber"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "gradio>=4.44", "soundfile", "librosa",
                           "transformers", "edge-tts", "gTTS"])
    import os
    os.chdir("ai-video-dubber")
sys.path.insert(0, ".")
print("working dir:", pathlib.Path.cwd())
import tts_tester
print("backends:", len(tts_tester.BACKENDS))


In [ ]:
# ── 2 ▸ Dark glassmorphism UI — one tab per TTS model ─────────────────
import gradio as gr
import numpy as np
import soundfile as sf
import tempfile
import tts_tester
from tts_tester import BACKENDS, get_backend

CSS = """
body, .gradio-container { background:#0b0d12 !important; color:#e8ecf4; }
.gradio-container { max-width:1200px; margin:auto; }
.glass { background:rgba(255,255,255,0.05); border:1px solid rgba(255,255,255,0.10);
  border-radius:18px; padding:18px 20px; backdrop-filter:blur(18px); }
.card { background:linear-gradient(135deg, rgba(255,255,255,0.08), rgba(255,255,255,0.03));
  border:1px solid rgba(255,255,255,0.12); border-radius:16px; padding:16px 20px; }
.badge { display:inline-block; padding:2px 10px; border-radius:999px;
  background:rgba(99,102,241,0.25); border:1px solid rgba(99,102,241,0.5);
  font-size:12px; margin-right:6px; }
.stat { color:#9aa7bd; font-size:13px; }
.console textarea { background:#05070c !important; color:#9ff0c8 !important;
  font-family:ui-monospace,Menlo,monospace !important; font-size:12px !important; }
h1,h2,h3 { color:#eef2fa !important; }
"""

def info_html(info):
    clone = "🎙 clone" if info.clone else "🔊 fixed voice"
    return (f'<div class="card"><div style="font-size:20px;font-weight:600;">'
            f'{info.name}</div>'
            f'<div class="stat">by {info.creator} · {info.size} · {info.license}'
            f'</div>'
            f'<div style="margin:8px 0;"><span class="badge">{info.languages}</span>'
            f'<span class="badge">{clone}</span></div>'
            f'<div class="stat">{info.notes}</div>'
            f'<div style="margin-top:8px;"><a href="{info.repo}" target="_blank" '
            f'style="color:#93c5fd;">{info.repo}</a></div></div>')

# per-tab component registries
UI = {}

def _fmt_log(lines):
    return "\n".join(lines)

def make_load_fn(bid, con_lines):
    def _load():
        b = get_backend(bid)
        con_lines.clear()
        def log(m):
            con_lines.append(str(m))
        try:
            b.load(log)
            con_lines.append("✅ loaded")
        except Exception as e:
            con_lines.append(f"❌ load failed: {e}")
        return _fmt_log(con_lines)
    return _load

def make_synth_fn(bid, con_lines, text_c, voice_c, lang_c, ref_c, settings_cs):
    def _synth(text, voice, language, ref, *sargs):
        b = get_backend(bid)
        con_lines.append(f"▶ synthesizing: {text[:60]!r}")
        settings = {}
        for c, v in zip(settings_cs, sargs):
            settings[c["name"]] = v
        if not b.loaded:
            try:
                b.load(lambda m: con_lines.append(str(m)))
            except Exception as e:
                con_lines.append(f"❌ load failed: {e}")
                return _fmt_log(con_lines), None
        try:
            y, sr = b.synthesize(text, voice=voice, language=language,
                                 ref_audio=ref, settings=settings,
                                 log=lambda m: con_lines.append(str(m)))
            p = tempfile.mkstemp(suffix=".wav")[1]
            sf.write(p, np.asarray(y, dtype=np.float32), int(sr))
            con_lines.append(f"✅ {len(y)/sr:.2f}s @ {sr} Hz")
            return _fmt_log(con_lines), p
        except Exception as e:
            con_lines.append(f"❌ synth failed: {e}")
            return _fmt_log(con_lines), None
    return _synth

with gr.Blocks(css=CSS, title="TTS Models Tester") as demo:
    gr.HTML('<h1>🎛 TTS Models Tester</h1>'
            '<div class="stat">Compare TTS engines for Hindi dubbing · '
            'models load only when you click <b>Load</b></div>')
    with gr.Tabs():
        for bid, cls in BACKENDS.items():
            info = cls.info
            con_lines = []
            with gr.Tab(info.name):
                gr.HTML(info_html(info))
                with gr.Row():
                    with gr.Column(scale=5, elem_classes=["glass"]):
                        load_btn = gr.Button("⬇️ Load model")
                        text_c = gr.Textbox(label="Text to speak", lines=3,
                                            value="नमस्ते, यह एक परीक्षण है।")
                        row = gr.Row()
                        with row:
                            voice_c = (gr.Dropdown(choices=cls().voices(),
                                                   label="Voice",
                                                   value=(cls().voices() or [None])[0])
                                       if cls().voices() else
                                       gr.Dropdown(choices=[], label="Voice",
                                                   visible=False))
                            lang_c = gr.Dropdown(choices=cls().languages(),
                                                 label="Language",
                                                 value=cls().languages()[0])
                        ref_c = gr.Audio(label="Reference clip (for cloning)",
                                         type="filepath",
                                         visible=info.clone)
                        settings_cs = []
                        for s in cls().settings_schema():
                            if s["type"] == "slider":
                                settings_cs.append(gr.Slider(
                                    minimum=s.get("min", 0), maximum=s.get("max", 2),
                                    value=s.get("value", 0), label=s["label"]))
                            else:
                                settings_cs.append(gr.Textbox(
                                    label=s["label"], value=s.get("value", "")))
                        synth_btn = gr.Button("🔊 Synthesize", variant="primary")
                        audio_out = gr.Audio(label="Output")
                    with gr.Column(scale=4, elem_classes=["glass"]):
                        gr.Markdown("**Console**")
                        con_c = gr.Textbox(lines=14, max_lines=24,
                                           interactive=False, show_copy_button=True,
                                           elem_classes=["console"])
                load_btn.click(make_load_fn(bid, con_lines), inputs=None,
                               outputs=[con_c])
                synth_btn.click(
                    make_synth_fn(bid, con_lines, text_c, voice_c, lang_c, ref_c,
                                  settings_cs),
                    inputs=[text_c, voice_c, lang_c, ref_c, *settings_cs],
                    outputs=[con_c, audio_out])

demo.launch(share=True, debug=False)
